In [1]:
%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Python312\Lib\urllib\parse.py' for module 'urllib.parse': UnicodeDecodeError
Traceback (most recent call last):
  File "d:\front-desk\myenv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\front-desk\myenv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python312\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1324, in _find_and_load_unlocked
ModuleNotFoundError: No module named 'autoreload'

During handling of the ab

In [11]:
from pydantic import BaseModel
from typing import List, Optional, Dict, Any

In [2]:
from utils import Vector_store_service, DocxProcessor, PDFprocessor

In [3]:
vector_store = Vector_store_service("550e8400-e29b-41d4-a716-446655440000")
docx_processor = DocxProcessor()
pdf_processor = PDFprocessor()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 582.99it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
# chunks1 = docx_processor.process_docx("./data/sample.docx")
chunks2 = pdf_processor.processPDF("./data/policy.pdf")

In [5]:
len(chunks2)

139

In [6]:
vector_store.embed_documents(chunks2)

{'status': 'success', 'embedded_count': 139}

In [17]:
collection = vector_store.get_collection()
print(collection.count())

4838


In [18]:
results = vector_store.retrieve_documents("learning autonomy according to holec")

In [19]:
print(results)

{'status': 'success', 'query': 'learning autonomy according to holec', 'total_retrieved': 5, 'filtered_count': 5, 'results': [{'id': 'b79a71bf-72ce-4b36-9272-46729ebdd725', 'content': 'learning” (Holec, 1981, p. 3). Holec’s idea of autonomy encompasses some components and capacities', 'metadata': {'creationdate': '2018-03-05T09:43:57+01:00', 'page': 2, 'creator': 'Microsoft® Word 2016', 'char_count': 4450, 'total_pages': 11, 'source': './data/document.pdf', 'chunk_method': 'smart_pdf_processor', 'producer': 'Microsoft® Word 2016', 'page_label': '2', 'author': 'agimeno', 'moddate': '2018-03-12T10:24:10-04:00'}, 'distance': 0.16542423}, {'id': '9d348156-2b1d-42ea-8243-05a24e428b89', 'content': 'of one’s own learning” (Holec, 1981, p. 3). Holec’s idea of autonomy encompasses some components', 'metadata': {'chunk_method': 'smart_pdf_processor', 'source': './data/document.pdf', 'creationdate': '2018-03-05T09:43:57+01:00', 'moddate': '2018-03-12T10:24:10-04:00', 'page_label': '2', 'producer'

In [3]:
from utils import memory

state = {"messages": []}
state["messages"] = memory.add_to_history(state, "Hello", "Hi there!")
print(memory.get_history(state, max_turns=2))

[{'role': 'user', 'content': 'Hello'}, {'role': 'assistant', 'content': 'Hi there!'}]


In [1]:
from model import llm
from node import intent_router_node

test_messages = [
    "hi",
    "hello how are you?",
    "What is learner autonomy?",
    "I want to book a room",
    "The AC is broken",
    "I need to speak to a human",
    "what did i ask you?",
    "holec",
]

for msg in test_messages:
    result = intent_router_node({"message": msg}, llm)
    print(f"'{msg}' -> {result.get('intent')}")



'hi' -> INFORMATION_RETRIEVAL
'hello how are you?' -> INFORMATION_RETRIEVAL
'What is learner autonomy?' -> INFORMATION_RETRIEVAL
'I want to book a room' -> INFORMATION_RETRIEVAL
'The AC is broken' -> INFORMATION_RETRIEVAL
'I need to speak to a human' -> INFORMATION_RETRIEVAL
'what did i ask you?' -> INFORMATION_RETRIEVAL
'holec' -> INFORMATION_RETRIEVAL


In [2]:
prompt = """Classify the user intent into one of these categories:
- INFORMATION_RETRIEVAL: Questions about information, facts, documents
- LEAD_CAPTURE: Booking requests, reservations, inquiries about services
- ISSUE_COMPLAINT: Problems, complaints, things not working
- HANDOFF_REQUEST: Requests to speak to human, manager, agent
- CHAT: Greetings, casual conversation, general chat

User message: "hi"

Respond with only the category name, nothing else."""

response = llm.invoke(prompt)
print(response.content)

CHAT


In [3]:
prompt2 = """Classify the user intent into one of these categories:
- INFORMATION_RETRIEVAL: Questions about information, facts, documents
- LEAD_CAPTURE: Booking requests, reservations, inquiries about services
- ISSUE_COMPLAINT: Problems, complaints, things not working
- HANDOFF_REQUEST: Requests to speak to human, manager, agent
- CHAT: Greetings, casual conversation, general chat

User message: "I want to book a room"

Respond with only the category name, nothing else."""

response = llm.invoke(prompt2)
print(response.content)

LEAD_CAPTURE


In [4]:
from model import llm
from node import intent_router_node

# Test with debug
state = {"message": "hi"}
result = intent_router_node(state, llm)

print("Full result:", result)
print("Intent:", result.get("intent"))
print("Type:", type(result.get("intent")))

Full result: {'message': 'hi', 'intent': 'INFORMATION_RETRIEVAL', 'confidence': 0.8}
Intent: INFORMATION_RETRIEVAL
Type: <class 'str'>


In [5]:
prompt = """Classify the user intent into one of these categories:
- INFORMATION_RETRIEVAL: Questions about information, facts, documents
- LEAD_CAPTURE: Booking requests, reservations, inquiries about services
- ISSUE_COMPLAINT: Problems, complaints, things not working
- HANDOFF_REQUEST: Requests to speak to human, manager, agent
- CHAT: Greetings, casual conversation, general chat

User message: "hi"

Respond with only the category name, nothing else."""

response = llm.invoke(prompt)
print("Direct LLM says:", response.content)

Direct LLM says: CHAT


In [6]:
import inspect
from node.router import intent_router_node
print(inspect.getsource(intent_router_node))

def intent_router_node(state: dict, llm):
    message = state["message"].lower()

    # 🚨 1. EMERGENCY
    EMERGENCY = ["fire", "smoke", "gas", "bleeding", "danger", "emergency"]
    if any(k in message for k in EMERGENCY):
        return {
            **state,
            "intent": "ISSUE_COMPLAINT",
            "confidence": 1.0,
        }

    # 🧯 2. ROOM / SERVICE ISSUES (FIXED)
    ROOM_CONTEXT = [
        "room", "bathroom", "toilet", "washroom",
        "fan", "ac", "light", "tv", "bed"
    ]

    ISSUE_SIGNALS = [
        "missing", "not working", "broken",
        "doesn't work", "does not work",
        "stopped working", "damaged", "leaking",
        "no"
    ]

    # 🧼 2. HYGIENE / HOUSEKEEPING ISSUES (CRITICAL)
    HYGIENE_ISSUE_KEYWORDS = [
        "dirty", "not cleaned", "unclean", "filthy",
        "condoms", "used condoms", "trash", "garbage",
        "leftover", "stains", "smell", "bad smell",
        "hygiene", "cleaning", "housekeeping"
    ]

    ARRIVAL_CONTEXT = 

In [1]:
from model import llm
from node import intent_router_node

test_messages = [
    "hi",
    "hello how are you?",
    "What is learner autonomy?",
    "I want to book a room",
    "The AC is broken",
    "I need to speak to a human",
]

for msg in test_messages:
    result = intent_router_node({"message": msg}, llm)
    print(f"'{msg}' -> {result.get('intent')}")

d:\front-desk\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'hi' -> CHAT
'hello how are you?' -> CHAT
'What is learner autonomy?' -> INFORMATION_RETRIEVAL
'I want to book a room' -> LEAD_CAPTURE
'The AC is broken' -> ISSUE_COMPLAINT
'I need to speak to a human' -> HANDOFF_REQUEST


In [2]:
from model import llm
from node import chat_node

state = {
    "message": "Hello, how are you?",
    "intent": "CHAT"
}

result = chat_node(state, llm)
print("Chat response:", result.get("result", {}).get("response"))

Chat response: I'm doing great, thanks for asking. Welcome to our hotel. How can I make your stay with us wonderful today? Do you need any assistance with checking in, or perhaps some recommendations for local attractions?


In [3]:
from utils import memory

state = {"messages": []}
state["messages"] = memory.add_to_history(state, "What is learner autonomy?", "Learner autonomy is the ability to take charge of one's own learning.")
state["messages"] = memory.add_to_history(state, "Tell me more", "It was defined by Holec in 1981.")

print("History:")
print(memory.get_history(state, max_turns=5))

History:
[{'role': 'user', 'content': 'What is learner autonomy?'}, {'role': 'assistant', 'content': "Learner autonomy is the ability to take charge of one's own learning."}, {'role': 'user', 'content': 'Tell me more'}, {'role': 'assistant', 'content': 'It was defined by Holec in 1981.'}]


In [4]:
from graph import graph

thread_id = "test-session-1"
org_id = "test-org-1"

# First question
r1 = graph.invoke(
    {"message": "What is learner autonomy?", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)
print("Q1:", r1.get("result", {}).get("response"))

# Follow-up - test memory
r2 = graph.invoke(
    {"message": "What did I just ask you?", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)
print("Q2:", r2.get("result", {}).get("response"))

# Chat test
r3 = graph.invoke(
    {"message": "hi", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)
print("Q3:", r3.get("result", {}).get("response"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 647.80it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q1: The provided context does not explicitly define learner autonomy, but it mentions "the context of learner autonomy" and "a learner can display more or less autonomy in different learning circumstances." This suggests that learner autonomy refers to the ability of a learner to take control of their own learning, but a clear definition is not provided.
Q2: I don't have that information. Let me connect you to a human agent.
Q3: Hello! Welcome to our hotel. How can I assist you today? Do you need help with checking in, or perhaps you have a question about our amenities?


In [5]:
from graph import graph

thread_id = "test-session-2"
org_id = "test-org-1"

# First message
r1 = graph.invoke(
    {"message": "What is learner autonomy?", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)
print("Q1:", r1.get("result", {}).get("response"))

# Check the state - does it have messages history?
print("\nFull state keys:", r1.keys())
print("Messages in state:", r1.get("messages", "NO MESSAGES KEY"))

Q1: The provided context doesn't explicitly define learner autonomy, but it mentions that a learner can display more or less autonomy in different learning circumstances, and it's discussed in the context of studies and limitations. If you're looking for a specific definition, I don't have that information. Let me connect you to a human agent.

Full state keys: dict_keys(['message', 'messages', 'org_id', 'intent', 'confidence', 'result'])
Messages in state: [{'role': 'user', 'content': 'What is learner autonomy?'}, {'role': 'assistant', 'content': "The provided context doesn't explicitly define learner autonomy, but it mentions that a learner can display more or less autonomy in different learning circumstances, and it's discussed in the context of studies and limitations. If you're looking for a specific definition, I don't have that information. Let me connect you to a human agent."}]


In [6]:
from graph import graph

thread_id = "test-session-3"
org_id = "test-org-1"

# First message
r1 = graph.invoke(
    {"message": "What is learner autonomy?", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)
print("Q1 messages count:", len(r1.get("messages", [])))

# Second message
r2 = graph.invoke(
    {"message": "Tell me more", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)
print("Q2 messages count:", len(r2.get("messages", [])))
print("Q2 messages:", r2.get("messages"))

Q1 messages count: 2
Q2 messages count: 4
Q2 messages: [{'role': 'user', 'content': 'What is learner autonomy?'}, {'role': 'assistant', 'content': 'The provided context does not give a clear definition of learner autonomy, but it mentions that a learner can display more or less autonomy in different learning circumstances, and that autonomy is being discussed in the context of a study.'}, {'role': 'user', 'content': 'Tell me more'}, {'role': 'assistant', 'content': "I don't have that information. Let me connect you to a human agent."}]


In [8]:
from utils import memory

# Simulate state with history
state = {
    "messages": [
        {"role": "user", "content": "What is learner autonomy?"},
        {"role": "assistant", "content": "Learner autonomy is the ability to take charge of one's own learning."},
    ]
}

history = memory.get_history(state, max_turns=4)
print("History:", history)

formatted = memory.format_for_llm(history)
print("Formatted:", formatted)

messages = memory.build_messages(
    query="Tell me more",
    system_prompt="You are a helpful assistant.",
    history=formatted,
    context="Some context here."
)
print("Messages:", messages)

History: [{'role': 'user', 'content': 'What is learner autonomy?'}, {'role': 'assistant', 'content': "Learner autonomy is the ability to take charge of one's own learning."}]
Formatted: [{'role': 'user', 'content': 'What is learner autonomy?'}, {'role': 'assistant', 'content': "Learner autonomy is the ability to take charge of one's own learning."}]
Messages: [{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'system', 'content': 'Knowledge Base Context:\nSome context here.'}, {'role': 'user', 'content': 'What is learner autonomy?'}, {'role': 'assistant', 'content': "Learner autonomy is the ability to take charge of one's own learning."}, {'role': 'user', 'content': 'Tell me more'}]


In [1]:
from graph import graph

thread_id = "test-session-4"
org_id = "test-org-1"

r1 = graph.invoke(
    {"message": "What is learner autonomy?", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)
print("Q1:", r1.get("result", {}).get("response"))

r2 = graph.invoke(
    {"message": "Tell me more", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)
print("Q2:", r2.get("result", {}).get("response"))

r3 = graph.invoke(
    {"message": "What did I just ask?", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)
print("Q3:", r3.get("result", {}).get("response"))

d:\front-desk\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 664.50it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Q1: The provided context doesn't give a direct definition of learner autonomy. However, it mentions that a learner can display more or less autonomy in different learning circumstances, and autonomy is discussed in the context of studies and learning circumstances. 

I don't have a clear definition of learner autonomy in the provided context. Let me connect you to a human agent.
Q2: I don't have that information. Let me connect you to a human agent.
Q3: You asked me to "Tell me more", but I don't have any additional information to provide based on our previous conversation. Let me connect you to a human agent.


In [2]:
from node.rag import rag_node

rag = rag_node()

# Simulate state WITH history (like Q2 would have)
state = {
    "message": "Tell me more",
    "org_id": "test-org-1",
    "messages": [
        {"role": "user", "content": "What is learner autonomy?"},
        {"role": "assistant", "content": "Learner autonomy is the ability to take charge of one's own learning."},
    ]
}

# Test query enhancement
enhanced = rag.enhance_query_with_history("Tell me more", state)
print("Enhanced query:", enhanced)

Enhanced query: What is learner autonomy? Tell me more


In [3]:
from graph import graph

thread_id = "test-session-5"
org_id = "test-org-1"

r1 = graph.invoke(
    {"message": "What is learner autonomy?", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)
print("Q1 messages:", r1.get("messages"))

# Now check if messages persist
r2 = graph.invoke(
    {"message": "Tell me more", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)
print("Q2 messages:", r2.get("messages"))
print("Q2 response:", r2.get("result", {}).get("response"))

Q1 messages: [{'role': 'user', 'content': 'What is learner autonomy?'}, {'role': 'assistant', 'content': "The provided context doesn't give a direct definition of learner autonomy. I don't have that information. Let me connect you to a human agent."}]
Q2 messages: [{'role': 'user', 'content': 'What is learner autonomy?'}, {'role': 'assistant', 'content': "The provided context doesn't give a direct definition of learner autonomy. I don't have that information. Let me connect you to a human agent."}, {'role': 'user', 'content': 'Tell me more'}, {'role': 'assistant', 'content': "The provided context doesn't provide more information about learner autonomy beyond mentioning that it can vary in different learning circumstances and that a learner can display more or less autonomy. I don't have that information. Let me connect you to a human agent."}]
Q2 response: The provided context doesn't provide more information about learner autonomy beyond mentioning that it can vary in different learni

In [11]:
from graph import graph

thread_id = "eval-session-1"
org_id = "test-org-2"

test_questions = [
    "What time is check-in?",
    "What is the cancellation policy?",
    "Do you allow pets?",
    "What amenities are included?",
    "What is the smoking policy?",
]

for q in test_questions:
    result = graph.invoke(
        {"message": q, "org_id": org_id},
        config={"configurable": {"thread_id": thread_id}}
    )
    print(f"Q: {q}")
    print(f"A: {result.get('result', {}).get('response')}")
    print("-" * 50)

lead node executed
Q: What time is check-in?
A: Our standard check-in time is 3:00 PM. However, if you'd like to arrive earlier, we can try to accommodate you based on availability. We also offer an early check-in option for a small fee, which would allow you to access your room as early as 12:00 PM.

May I ask, do you have a reservation with us already, or would you like to book a room? If so, I'd be happy to look up your reservation and provide you with more details. Could you please provide me with your name or reservation number?
--------------------------------------------------
rag node executed
Q: What is the cancellation policy?
A: I don't have that information. Let me connect you to a human agent.
--------------------------------------------------
rag node executed
Q: Do you allow pets?
A: No, pets are not allowed, with the exception of guide dogs or assistance dogs for people with visual impairment.
--------------------------------------------------
rag node executed
Q: What 

In [ ]:
from graph import graph

thread_id = "eval-session-2"
org_id = "test-org-2"

test_questions = [
    "i want to know the checkin time",
    "i want to know the cancelation policy",
    "Do you allow pets?",
    "What amenities are included?",
    "What is the smoking policy?",
]

for q in test_questions:
    result = graph.invoke(
        {"message": q, "org_id": org_id},
        config={"configurable": {"thread_id": thread_id}}
    )
    print(f"Q: {q}")
    print(f"A: {result.get('result', {}).get('response')}")
    print("-" * 50)

rag node executed
Q: i want to know the checkin time
A: Check-in takes place from 14:00 and until 20:00 on the day of the client's arrival. However, if the guest arrives before Check-in time and the Hotel has rooms available, they can enter as early as 7:00.
--------------------------------------------------
rag node executed
Q: i want to know the cancelation policy
A: I don't have that information. Let me connect you to a human agent.
--------------------------------------------------
rag node executed
Q: Do you allow pets?
A: No, pets are not allowed, with the exception of guide dogs or assistance dogs for people with visual impairment.
--------------------------------------------------
rag node executed
Q: What amenities are included?
A: I don't have that information. Let me connect you to a human agent.
--------------------------------------------------
rag node executed
Q: What is the smoking policy?
A: The Hotel is a 100% smoke-free environment. Smoking is prohibited throughout t

In [1]:
from db import engine, get_db, SessionLocal
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print(result.scalar())


1


In [2]:
from sqlalchemy import inspect

inspector = inspect(engine)
print(inspector.get_table_names())


['users']


In [3]:
from db import Base
Base.metadata.create_all(engine)


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verify DATABASE_URL is set
print(f"DATABASE_URL set: {'DATABASE_URL' in os.environ}")

DATABASE_URL set: True


In [3]:
from db.connection import engine
from db.models import Base

# Create all tables
Base.metadata.create_all(bind=engine)
print("✅ Database tables created successfully!")

✅ Database tables created successfully!


In [4]:
from db import get_db, Conversation, Message
import uuid

with get_db() as db:
    # Test query
    count = db.query(Conversation).count()
    print(f"✅ Database connection working! Conversations: {count}")

✅ Database connection working! Conversations: 0


In [1]:
from utils import PostgresCheckpointer

checkpointer = PostgresCheckpointer()
print("✅ PostgresCheckpointer initialized!")

d:\front-desk\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ PostgresCheckpointer initialized!


In [1]:
# Restart kernel first, then:
from graph import graph
import uuid

thread_id = str(uuid.uuid4())
org_id = str(uuid.uuid4())

result = graph.invoke(
    {"message": "Hello, I want to book a room", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)

print(f"Intent: {result.get('intent')}")
print(f"Response: {result.get('result', {}).get('response')}")

d:\front-desk\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NoForeignKeysError: Could not determine join condition between parent/child tables on relationship Organization.conversations - there are no foreign keys linking these tables.  Ensure that referencing columns are associated with a ForeignKey or ForeignKeyConstraint, or specify a 'primaryjoin' expression.

In [1]:
# Cell 1: Recreate tables
from db import engine, Base

Base.metadata.drop_all(bind=engine)
Base.metadata.create_all(bind=engine)
print("✅ Tables recreated!")

✅ Tables recreated!


In [3]:
# Cell 2: Create test org and test the graph
from db import get_db
from db.models import Organization
from graph import graph
import uuid

# Create test organization (required due to FK)
org_id = str(uuid.uuid4())
with get_db() as db:
    org = Organization(id=uuid.UUID(org_id), name="Test Hotel", slug="test-hotel")
    db.add(org)

# Test the graph
thread_id = str(uuid.uuid4())
result = graph.invoke(
    {"message": "Hello, I want to book a room", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)

print(f"Intent: {result.get('intent')}")
print(f"Response: {result.get('result', {}).get('response')}")

lead node executed


TypeError: PostgresCheckpointer.put() takes 4 positional arguments but 5 were given

In [1]:
from db import engine, Base
from db.models import Organization
from db import get_db
from graph import graph
import uuid

# Recreate tables
Base.metadata.drop_all(bind=engine)
Base.metadata.create_all(bind=engine)
print("✅ Tables recreated!")

# Create test org
org_id = str(uuid.uuid4())
with get_db() as db:
    org = Organization(id=uuid.UUID(org_id), name="Test Hotel", slug="test-hotel")
    db.add(org)

# Test graph
thread_id = str(uuid.uuid4())
result = graph.invoke(
    {"message": "Hello, I want to book a room", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)

print(f"Intent: {result.get('intent')}")
print(f"Response: {result.get('result', {}).get('response')}")

d:\front-desk\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Tables recreated!
lead node executed


AttributeError: 'JsonPlusSerializer' object has no attribute 'dumps'

In [1]:
from db import engine, Base
from db.models import Organization
from db import get_db
from graph import graph
import uuid

# Recreate tables
Base.metadata.drop_all(bind=engine)
Base.metadata.create_all(bind=engine)
print("✅ Tables recreated!")

# Create test org
org_id = str(uuid.uuid4())
with get_db() as db:
    org = Organization(id=uuid.UUID(org_id), name="Test Hotel", slug="test-hotel")
    db.add(org)

# Test graph
thread_id = str(uuid.uuid4())
result = graph.invoke(
    {"message": "Hello, I want to book a room", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)

print(f"Intent: {result.get('intent')}")
print(f"Response: {result.get('result', {}).get('response')}")

d:\front-desk\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Tables recreated!
lead node executed


NotImplementedError: 

In [1]:
from db import engine, Base
from db.models import Organization
from db import get_db
from graph import graph
import uuid

# Recreate tables
Base.metadata.drop_all(bind=engine)
Base.metadata.create_all(bind=engine)
print("✅ Tables recreated!")

# Create test org
org_id = str(uuid.uuid4())
with get_db() as db:
    org = Organization(id=uuid.UUID(org_id), name="Test Hotel", slug="test-hotel")
    db.add(org)

# Test graph
thread_id = str(uuid.uuid4())
result = graph.invoke(
    {"message": "Hello, I want to book a room", "org_id": org_id},
    config={"configurable": {"thread_id": thread_id}}
)

print(f"Intent: {result.get('intent')}")
print(f"Response: {result.get('result', {}).get('response')}")

d:\front-desk\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Tables recreated!
lead node executed
Intent: LEAD_CAPTURE
Response: Welcome to our hotel. I'd be happy to help you book a room. Before we get started, may I ask for your name, please? And could you also please tell me what dates you're looking to stay with us?

Additionally, what type of room are you looking for? We have a variety of options, including single, double, and suites. Do you have any specific preferences, such as a king-sized bed or a room with a view?

We also offer some great packages and deals, so I'd be happy to let you know what's available during your stay. Shall I go ahead and check availability for you?


In [7]:
# Cell 4: Interactive chat loop (optional)
from graph import graph
import uuid

org_id = "550e8400-e29b-41d4-a716-446655440000"
thread_id = str(uuid.uuid4())

print("Hotel Front Desk Bot (type 'quit' to exit)")
print(f"Org ID: {org_id[:8]}...")
print(f"Thread: {thread_id[:8]}...")
print("-" * 40)

while True:
    message = input("\nYou: ").strip()
    if not message:
        continue
    if message.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    
    result = graph.invoke(
        {"message": message, "org_id": org_id},
        config={"configurable": {"thread_id": thread_id}}
    )
    
    response = result.get("result", {}).get("response", "No response")
    print(f"\nBot: {response}")

Hotel Front Desk Bot (type 'quit' to exit)
Org ID: 550e8400...
Thread: ba804bfa...
----------------------------------------


IntegrityError: (psycopg2.errors.ForeignKeyViolation) insert or update on table "conversations" violates foreign key constraint "conversations_org_id_fkey"
DETAIL:  Key (org_id)=(550e8400-e29b-41d4-a716-446655440000) is not present in table "organizations".

[SQL: INSERT INTO conversations (id, org_id, visitor_id, visitor_name, visitor_metadata, state, agent_id, handoff_reason, created_at, updated_at, connected_at, closed_at) VALUES (%(id)s::UUID, %(org_id)s::UUID, %(visitor_id)s::UUID, %(visitor_name)s, %(visitor_metadata)s::JSON, %(state)s, %(agent_id)s::UUID, %(handoff_reason)s, %(created_at)s, %(updated_at)s, %(connected_at)s, %(closed_at)s)]
[parameters: {'id': UUID('98aa2990-b7e3-4ff0-a197-2df82db7fc8e'), 'org_id': UUID('550e8400-e29b-41d4-a716-446655440000'), 'visitor_id': UUID('cee0e502-728e-4c2b-a05c-043613939f6e'), 'visitor_name': None, 'visitor_metadata': '{}', 'state': 'AI_ACTIVE', 'agent_id': None, 'handoff_reason': None, 'created_at': datetime.datetime(2026, 2, 15, 3, 30,

In [8]:
# Cell 1: Create the organization first
from db import get_db
from db.models import Organization
import uuid

org_id = "550e8400-e29b-41d4-a716-446655440000"
org_uuid = uuid.UUID(org_id)

with get_db() as db:
    # Check if org exists
    existing = db.query(Organization).filter(Organization.id == org_uuid).first()
    if not existing:
        org = Organization(
            id=org_uuid,
            name="Test Hotel",
            slug="test-hotel"
        )
        db.add(org)
        db.commit()
        print(f"✅ Created organization: {org.name} ({org.id})")
    else:
        print(f"✅ Organization already exists: {existing.name} ({existing.id})")

IntegrityError: (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "organizations_slug_key"
DETAIL:  Key (slug)=(test-hotel) already exists.

[SQL: INSERT INTO organizations (id, name, slug, config, plan, status, created_at) VALUES (%(id)s::UUID, %(name)s, %(slug)s, %(config)s::JSON, %(plan)s, %(status)s, %(created_at)s)]
[parameters: {'id': UUID('550e8400-e29b-41d4-a716-446655440000'), 'name': 'Test Hotel', 'slug': 'test-hotel', 'config': '{}', 'plan': 'FREE', 'status': 'ACTIVE', 'created_at': datetime.datetime(2026, 2, 15, 3, 31, 39, 886606, tzinfo=datetime.timezone.utc)}]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [9]:
# Cell 2: Now test the graph
from graph import graph
import uuid

org_id = "550e8400-e29b-41d4-a716-446655440000"
thread_id = str(uuid.uuid4())

print("Hotel Front Desk Bot (type 'quit' to exit)")
print(f"Org ID: {org_id[:8]}...")
print(f"Thread: {thread_id[:8]}...")
print("-" * 40)

while True:
    message = input("\nYou: ").strip()
    if not message:
        continue
    if message.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    
    result = graph.invoke(
        {"message": message, "org_id": org_id},
        config={"configurable": {"thread_id": thread_id}}
    )
    
    response = result.get("result", {}).get("response", "No response")
    print(f"\nBot: {response}")
    

Hotel Front Desk Bot (type 'quit' to exit)
Org ID: 550e8400...
Thread: 2bf10265...
----------------------------------------


IntegrityError: (psycopg2.errors.ForeignKeyViolation) insert or update on table "conversations" violates foreign key constraint "conversations_org_id_fkey"
DETAIL:  Key (org_id)=(550e8400-e29b-41d4-a716-446655440000) is not present in table "organizations".

[SQL: INSERT INTO conversations (id, org_id, visitor_id, visitor_name, visitor_metadata, state, agent_id, handoff_reason, created_at, updated_at, connected_at, closed_at) VALUES (%(id)s::UUID, %(org_id)s::UUID, %(visitor_id)s::UUID, %(visitor_name)s, %(visitor_metadata)s::JSON, %(state)s, %(agent_id)s::UUID, %(handoff_reason)s, %(created_at)s, %(updated_at)s, %(connected_at)s, %(closed_at)s)]
[parameters: {'id': UUID('c94e966f-512b-4ff3-a1ff-079e66cf5307'), 'org_id': UUID('550e8400-e29b-41d4-a716-446655440000'), 'visitor_id': UUID('c19598fc-0daf-494a-a4f2-e9ae48c31ee3'), 'visitor_name': None, 'visitor_metadata': '{}', 'state': 'AI_ACTIVE', 'agent_id': None, 'handoff_reason': None, 'created_at': datetime.datetime(2026, 2, 15, 3, 32,

In [10]:
# Cell 1: Get or update the organization
from db import get_db
from db.models import Organization
import uuid

org_id = "550e8400-e29b-41d4-a716-446655440000"
org_uuid = uuid.UUID(org_id)

with get_db() as db:
    # Check by ID first
    org_by_id = db.query(Organization).filter(Organization.id == org_uuid).first()
    
    # Check by slug
    org_by_slug = db.query(Organization).filter(Organization.slug == "test-hotel").first()
    
    if org_by_id:
        print(f"✅ Organization exists with correct ID: {org_by_id.name} ({org_by_id.id})")
        org_id = str(org_by_id.id)
    elif org_by_slug:
        # Use existing org's ID instead
        print(f"⚠️ Organization 'test-hotel' exists with different ID: {org_by_slug.id}")
        print(f"   Using existing org_id: {org_by_slug.id}")
        org_id = str(org_by_slug.id)
    else:
        # Create new
        org = Organization(
            id=org_uuid,
            name="Test Hotel",
            slug="test-hotel"
        )
        db.add(org)
        db.commit()
        print(f"✅ Created organization: {org.name} ({org.id})")

print(f"\n📌 Use this org_id: {org_id}")

⚠️ Organization 'test-hotel' exists with different ID: 65303a42-3b19-4d67-8450-9dadbd91b17f
   Using existing org_id: 65303a42-3b19-4d67-8450-9dadbd91b17f

📌 Use this org_id: 65303a42-3b19-4d67-8450-9dadbd91b17f


In [11]:
# Cell 2: Test the graph with correct org_id
from graph import graph
import uuid

# Use the org_id from Cell 1 output
org_id = "65303a42-3b19-4d67-8450-9dadbd91b17f"  # Replace with actual org_id

thread_id = str(uuid.uuid4())

print("Hotel Front Desk Bot (type 'quit' to exit)")
print(f"Org ID: {org_id[:8]}...")
print(f"Thread: {thread_id[:8]}...")
print("-" * 40)

while True:
    message = input("\nYou: ").strip()
    if not message:
        continue
    if message.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    
    result = graph.invoke(
        {"message": message, "org_id": org_id},
        config={"configurable": {"thread_id": thread_id}}
    )
    
    response = result.get("result", {}).get("response", "No response")
    print(f"\nBot: {response}")

Hotel Front Desk Bot (type 'quit' to exit)
Org ID: 65303a42...
Thread: b005868b...
----------------------------------------
chat node executed


DetachedInstanceError: Instance <Conversation at 0x17b3bd9b800> is not bound to a Session; attribute refresh operation cannot proceed (Background on this error at: https://sqlalche.me/e/20/bhk3)